In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import time

sys.path.append('../src')

In [3]:
from llm_val.model.BaseGigaModel import BaseGigaModel
from llm_val.model.MultiThreadGigaModel import MultiThreadGigaModel

In [4]:
RQUID = 'INSERT_YOUR_CLIENT_ID_HERE'
AUTH_DATA = 'INSERT_YOUR_CLIENT_SECRET_HERE'

In [5]:
OAUTH_URL = 'https://ngw.devices.sberbank.ru:9443/api/v2/oauth' #MLSpace
API_URL = 'https://gigachat.devices.sberbank.ru/api/v1' #MLSpace
TOKEN = None #MLSpace

In [6]:
# OAUTH_URL = 'https://sm-auth-sd.prom-88-89-apps.ocp-geo.ocp.sigma.sbrf.ru/api/v2/oauth' #Sigma
# API_URL = 'https://gigachat.devices.sberbank.ru/api/v1' #Sigma
# TOKEN = None #Sigma

In [7]:
# OAUTH_URL = None #datalab
# API_URL = 'http://liveaccess/v1/gc' #datalab
# TOKEN = os.environ.get('JPY_API_TOKEN') #datalab

## GigaModel

Все обертки наследуются от абстрактного класса `GigaModel`, у которого есть конструктор, в котором создается экземпляр API и определяется системный промпт, и абстрактные методы `fit` и `predict`, которые необходимо определить в классе-обертке.

Обертка нужна для удобной работы с API.

Данная example-тетрадка показывает, что можно написать обертку под каждую задачу. Единственное, что необходимо сделать - реализовать методы `fit` и `predict`. Помимо методов, которые необходимо определить, все наследуемые классы наследуют от класса-родителя три аттрибута: экземпляр класса `GigaChatAPI`, название модели и системный промпт. Примеры реализаций можно посмотреть в `llm_val.model`

## BaseGigaModel

`BaseGigaModel` является примером обертки над `GigaChatAPI` для работы в удобном формате. Можно реализовывать свои обертки в зависимости от решаемой задачи, структуры входных данных и выходных данных. В данном случае `BaseGigaModel` для инференса принимает вопрос как одну строку или список вопросов и на выходе выдает список ответов по каждому вопросу.

In [8]:
base_model = BaseGigaModel(contour=None,
                           token=TOKEN,
                           auth_data=AUTH_DATA,
                           api_url=API_URL,
                           oauth_url=OAUTH_URL,
                           rquid=RQUID,
                           model='GigaChat:latest')

In [9]:
message = 'Кто действующий чемпион NBA?'

In [10]:
%%time
answer = base_model.predict(message)

CPU times: user 8.36 ms, sys: 4.14 ms, total: 12.5 ms
Wall time: 456 ms


In [11]:
messages = ['Сколько в среднем весит Капибара?',
            'Сколько лет живет капибара?',
            'Как звали капибару по имени Боб?',
            'Расскажи анекдот про капибар',
            'Как понять, что перед тобой стоит капибара?',
            'Напиши стихотворение про капибару']

In [12]:
%%time
answers = base_model.predict(messages)

CPU times: user 63.5 ms, sys: 9.65 ms, total: 73.2 ms
Wall time: 4.77 s


## MultiThreadGigaModel

`MultiThreadGigaModel` является примером обертки над `GigaChatAPI` c применением многопоточности. По своему функционалу это тот же `BaseGigaModel`, но с многопоточным инференсом.

In [13]:
multithread_model = MultiThreadGigaModel(contour=None,
                                         token=None,
                                         auth_data=AUTH_DATA,
                                         api_url=API_URL,
                                         oauth_url=OAUTH_URL,
                                         rquid=RQUID,
                                         model='GigaChat:latest',
                                         n_jobs=8)

In [14]:
messages = ['Сколько в среднем весит Капибара?',
            'Сколько лет живет капибара?',
            'Как звали капибару по имени Боб?',
            'Расскажи анекдот про капибар',
            'Как понять, что перед тобой стоит капибара?',
            'Напиши стихотворение про капибару']

In [15]:
%%time
answers = multithread_model.predict(messages)

CPU times: user 81.8 ms, sys: 6.51 ms, total: 88.3 ms
Wall time: 1.72 s


## Добавление истории диалога при предсказаниях модели

In [16]:
q = ['Какой вопрос я задавал в начале нашего диалога?',
     'Какой вопрос я задавал в начале нашего диалога?']

context = [[{'role': 'user',
            'content': 'Из каких химических элементов состоит соль поваренная?'},
           {'role': 'assistant',
            'content': 'Из натрия и хлора'}],
           [{'role': 'user',
             'content': 'Сколько примерно весит жерех?'},
            {'role': 'assistant',
             'content': 'Взрослый жерех весит в среднем 1.5-2 килограмма, однако крупные особи могут весить до 8 килограмм.'}]]

In [17]:
base_answers = base_model.predict(q, chat_history=context)

In [18]:
base_answers

['В начале нашего диалога ты спросил: *"Из каких химических элементов состоит соль поваренная?"*',
 'В начале нашего диалога ты спрашивал: *«Сколько примерно весит жерех?»*']

In [19]:
multi_answers = multithread_model.predict(q, chat_history=context)

In [20]:
multi_answers

['В начале нашего диалога ты спрашивал: *"Из каких химических элементов состоит соль поваренная?"*',
 'В начале нашего диалога ты спрашивал: *«Сколько примерно весит жерех?»*']